# 03 - Descomposición de Series Temporales

## Pregunta de negocio: ¿Hay vehículos perdiendo eficiencia?

---

### Fundamentos teóricos

#### Componentes de una serie temporal

Toda serie temporal puede descomponerse en tres componentes fundamentales:

1. **Tendencia (Trend)**: Movimiento de largo plazo. En nuestro caso, un incremento sostenido
   en el consumo indicaría degradación mecánica.

2. **Estacionalidad (Seasonality)**: Patrones cíclicos que se repiten en períodos fijos.
   Por ejemplo, mayor consumo los viernes (viajes largos de fin de semana) o en invierno
   (mayor uso de calefacción/batería).

3. **Residual (Noise)**: Variación aleatoria no explicada por tendencia ni estacionalidad.
   Picos inusuales pueden indicar eventos atípicos (accidentes, cargas pesadas, etc.).

#### Modelos de descomposición

- **Aditivo**: $Y_t = T_t + S_t + R_t$ (cuando la variación estacional es constante).
- **Multiplicativo**: $Y_t = T_t \times S_t \times R_t$ (cuando la variación estacional
  crece proporcionalmente con la tendencia).

#### Estacionariedad y test ADF

Una serie es **estacionaria** si sus propiedades estadísticas (media, varianza) no cambian
en el tiempo. El **test Augmented Dickey-Fuller (ADF)** evalúa:

- $H_0$: La serie tiene raíz unitaria (no estacionaria).
- $H_1$: La serie es estacionaria.

Si p-valor < 0.05, rechazamos $H_0$ y concluimos que la serie es estacionaria.
La **diferenciación** ($Y'_t = Y_t - Y_{t-1}$) es una técnica común para lograr estacionariedad.

#### Estadísticas rodantes (Rolling Statistics)

La media y desviación estándar móviles nos permiten visualizar cómo evoluciona el
comportamiento promedio y la variabilidad, suavizando el ruido diario.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

vtype_colors = {
    'electrico': '#2ecc71',
    'gasolina': '#3498db',
    'hibrido': '#9b59b6',
    'deportivo': '#e74c3c'
}

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
print(f"Directorio raíz del proyecto: {project_root}")

---
## 2. Carga y preparación de datos de telemetría

Cargamos todos los archivos de telemetría y los agregamos por día y vehículo.
Calculamos métricas diarias: consumo medio, velocidad media y temperatura media de batería.

In [ ]:
# Cargar todos los archivos de telemetría
telemetry_dir = os.path.join(project_root, "data/raw/telemetry")
telemetry_files = sorted(glob.glob(os.path.join(telemetry_dir, "telemetry_*.csv")))

print(f"Archivos de telemetría encontrados: {len(telemetry_files)}")
for f in telemetry_files[:5]:
    print(f"  {os.path.basename(f)}")
if len(telemetry_files) > 5:
    print(f"  ... y {len(telemetry_files) - 5} más")

# Concatenar
df_list = []
for f in telemetry_files:
    df_temp = pd.read_csv(f)
    df_list.append(df_temp)

df_telem = pd.concat(df_list, ignore_index=True)
print(f"\nTelemetría total: {df_telem.shape}")
print(f"Columnas: {df_telem.columns.tolist()}")

In [ ]:
# Parsear timestamps
df_telem['timestamp'] = pd.to_datetime(df_telem['timestamp'])
df_telem['date'] = df_telem['timestamp'].dt.date
df_telem['date'] = pd.to_datetime(df_telem['date'])

print(f"Rango temporal: {df_telem['timestamp'].min()} - {df_telem['timestamp'].max()}")
print(f"Días únicos: {df_telem['date'].nunique()}")
print(f"Vehículos únicos: {df_telem['vehicle_id'].nunique()}")

In [ ]:
# Agregar por día y vehículo
agg_dict = {}
if 'fuel_consumption_rate' in df_telem.columns:
    agg_dict['fuel_consumption_rate'] = 'mean'
if 'speed_kmh' in df_telem.columns:
    agg_dict['speed_kmh'] = 'mean'
if 'battery_temp_c' in df_telem.columns:
    agg_dict['battery_temp_c'] = 'mean'
if 'battery_soc_pct' in df_telem.columns:
    agg_dict['battery_soc_pct'] = 'mean'
if 'motor_power_kw' in df_telem.columns:
    agg_dict['motor_power_kw'] = 'mean'

# Agregar conteo de registros
agg_dict[df_telem.columns[0]] = 'count'  # usar primera columna para conteo

df_daily_vehicle = df_telem.groupby(['date', 'vehicle_id']).agg(agg_dict).reset_index()
count_col = df_telem.columns[0]
df_daily_vehicle = df_daily_vehicle.rename(columns={count_col: 'n_records'})

print(f"Datos diarios por vehículo: {df_daily_vehicle.shape}")
df_daily_vehicle.head()

In [ ]:
# Agregar a nivel de flota (promedio diario de toda la flota)
fleet_agg = {col: 'mean' for col in df_daily_vehicle.columns
             if col not in ['date', 'vehicle_id', 'n_records']}
fleet_agg['n_records'] = 'sum'

df_fleet = df_daily_vehicle.groupby('date').agg(fleet_agg).reset_index()
df_fleet = df_fleet.set_index('date').sort_index()

print(f"Serie temporal de flota: {df_fleet.shape}")
print(f"Rango: {df_fleet.index.min()} - {df_fleet.index.max()}")
df_fleet.head()

---
## 3. Serie temporal a nivel de flota

Visualizamos el consumo diario promedio de toda la flota con media móvil de 7 días
para identificar tendencias generales.

In [ ]:
# Determinar la columna de consumo
consumption_col = None
for col in ['fuel_consumption_rate', 'consumption_mean', 'avg_consumption']:
    if col in df_fleet.columns:
        consumption_col = col
        break

if consumption_col is None:
    # Usar la primera columna numérica disponible
    num_cols = [c for c in df_fleet.columns if c != 'n_records']
    consumption_col = num_cols[0] if num_cols else df_fleet.columns[0]
    print(f"Usando '{consumption_col}' como métrica principal.")
else:
    print(f"Columna de consumo: '{consumption_col}'")

# Media móvil y desviación estándar móvil
rolling_mean = df_fleet[consumption_col].rolling(window=7, min_periods=1).mean()
rolling_std = df_fleet[consumption_col].rolling(window=7, min_periods=1).std()

fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(df_fleet.index, df_fleet[consumption_col],
        color='#3498db', alpha=0.3, linewidth=0.8, label='Diario')
ax.plot(df_fleet.index, rolling_mean,
        color='#e74c3c', linewidth=2.5, label='Media móvil (7 días)')
ax.fill_between(df_fleet.index,
                rolling_mean - rolling_std,
                rolling_mean + rolling_std,
                alpha=0.15, color='#e74c3c', label='± 1 Std móvil')

ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel(f'{consumption_col}', fontsize=12)
ax.set_title('Consumo diario promedio de la flota', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"Consumo medio de la flota: {df_fleet[consumption_col].mean():.3f}")
print(f"Desviación estándar: {df_fleet[consumption_col].std():.3f}")

---
## 4. Descomposición estacional

Aplicamos `seasonal_decompose` de statsmodels con período=7 (semanal)
para separar tendencia, estacionalidad y residual.

In [ ]:
# Rellenar valores faltantes si existen (requerido por seasonal_decompose)
series = df_fleet[consumption_col].copy()
series = series.asfreq('D')  # Asegurar frecuencia diaria
series = series.interpolate(method='linear')  # Interpolar NaN

# Descomposición aditiva con período semanal
if len(series.dropna()) >= 14:  # Necesitamos al menos 2 periodos
    decomposition = seasonal_decompose(series, model='additive', period=7)

    fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)

    # Serie original
    axes[0].plot(decomposition.observed, color='#2c3e50', linewidth=1)
    axes[0].set_ylabel('Observado', fontsize=11)
    axes[0].set_title('Descomposición de Serie Temporal - Consumo de la Flota',
                      fontsize=14, fontweight='bold')

    # Tendencia
    axes[1].plot(decomposition.trend, color='#e74c3c', linewidth=2)
    axes[1].set_ylabel('Tendencia', fontsize=11)

    # Estacionalidad
    axes[2].plot(decomposition.seasonal, color='#2ecc71', linewidth=1)
    axes[2].set_ylabel('Estacionalidad', fontsize=11)

    # Residual
    axes[3].plot(decomposition.resid, color='#9b59b6', linewidth=0.8, alpha=0.7)
    axes[3].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[3].set_ylabel('Residual', fontsize=11)
    axes[3].set_xlabel('Fecha', fontsize=12)

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    print("Interpretación de componentes:")
    print(f"  Tendencia - Rango: [{decomposition.trend.min():.3f}, {decomposition.trend.max():.3f}]")
    trend_change = decomposition.trend.dropna()
    if len(trend_change) > 1:
        pct_change = (trend_change.iloc[-1] - trend_change.iloc[0]) / abs(trend_change.iloc[0]) * 100
        print(f"  Cambio de tendencia: {pct_change:+.2f}%")
    print(f"  Estacionalidad - Amplitud: {decomposition.seasonal.max() - decomposition.seasonal.min():.4f}")
    print(f"  Residual - Std: {decomposition.resid.std():.4f}")
else:
    print(f"Serie muy corta ({len(series)} días). Se necesitan al menos 14 días.")

---
## 5. Tendencias por vehículo

Descomponemos la serie de los 5 vehículos con más datos y comparamos
sus tendencias. Un vehículo con tendencia creciente en consumo podría
estar sufriendo degradación mecánica.

In [ ]:
# Identificar los 5 vehículos con más datos
vehicle_counts = df_daily_vehicle.groupby('vehicle_id')['date'].count()
top_vehicles = vehicle_counts.nlargest(5).index.tolist()

print("Top 5 vehículos con más datos diarios:")
for v in top_vehicles:
    print(f"  {v}: {vehicle_counts[v]} días")

In [ ]:
# Descomponer cada vehículo y extraer tendencia
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

vehicle_trends = {}
colors_v = plt.cm.tab10(np.linspace(0, 1, len(top_vehicles)))

for idx, vid in enumerate(top_vehicles):
    df_v = df_daily_vehicle[df_daily_vehicle['vehicle_id'] == vid].copy()
    df_v = df_v.set_index('date').sort_index()

    if consumption_col not in df_v.columns:
        # Buscar alternativa
        alt_cols = [c for c in df_v.columns if c not in ['vehicle_id', 'n_records']]
        if alt_cols:
            use_col = alt_cols[0]
        else:
            continue
    else:
        use_col = consumption_col

    series_v = df_v[use_col].copy()
    series_v = series_v.asfreq('D')
    series_v = series_v.interpolate(method='linear')

    # Serie original
    axes[0].plot(series_v.index, series_v.values,
                 alpha=0.4, linewidth=0.8, color=colors_v[idx])

    # Descomponer si hay suficientes datos
    if len(series_v.dropna()) >= 14:
        decomp_v = seasonal_decompose(series_v, model='additive', period=7)
        trend_v = decomp_v.trend.dropna()
        vehicle_trends[vid] = trend_v
        axes[1].plot(trend_v.index, trend_v.values,
                     linewidth=2.5, color=colors_v[idx], label=str(vid))
    else:
        # Usar media móvil como proxy
        rm = series_v.rolling(7, min_periods=1).mean()
        vehicle_trends[vid] = rm
        axes[1].plot(rm.index, rm.values,
                     linewidth=2.5, color=colors_v[idx], label=f"{vid} (rolling)",
                     linestyle='--')

axes[0].set_title('Consumo diario por vehículo (serie original)', fontsize=13, fontweight='bold')
axes[0].set_ylabel(use_col, fontsize=11)

axes[1].set_title('Tendencias de consumo por vehículo', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Tendencia', fontsize=11)
axes[1].set_xlabel('Fecha', fontsize=12)
axes[1].legend(title='Vehículo', fontsize=9, bbox_to_anchor=(1.05, 1))

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluar cuáles vehículos muestran tendencia creciente (degradación)
print("Análisis de tendencia por vehículo:")
print("=" * 60)

for vid, trend in vehicle_trends.items():
    trend_clean = trend.dropna()
    if len(trend_clean) > 1:
        # Regresión lineal simple para medir pendiente
        x = np.arange(len(trend_clean))
        slope, intercept = np.polyfit(x, trend_clean.values, 1)
        pct_change = (trend_clean.iloc[-1] - trend_clean.iloc[0]) / abs(trend_clean.iloc[0]) * 100

        status = "DEGRADACIÓN" if slope > 0 else "MEJORA"
        emoji_indicator = "[!]" if slope > 0 and abs(pct_change) > 5 else "[OK]"

        print(f"  {emoji_indicator} Vehículo {vid}: pendiente={slope:.5f}, "
              f"cambio={pct_change:+.2f}% -> {status}")

---
## 6. Test de estacionariedad (ADF)

Evaluamos si la serie de consumo de la flota es estacionaria. Si no lo es,
aplicamos diferenciación para lograrlo.

In [ ]:
def adf_test(series, title=''):
    """Realiza el test ADF e imprime resultados."""
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"Test ADF - {title}")
    print(f"  Estadístico ADF: {result[0]:.4f}")
    print(f"  p-valor: {result[1]:.6f}")
    print(f"  Lags usados: {result[2]}")
    print(f"  Observaciones: {result[3]}")
    print(f"  Valores críticos:")
    for key, value in result[4].items():
        print(f"    {key}: {value:.4f}")

    if result[1] < 0.05:
        print(f"  CONCLUSIÓN: Serie ESTACIONARIA (p < 0.05)")
    else:
        print(f"  CONCLUSIÓN: Serie NO estacionaria (p >= 0.05)")
    print()
    return result[1]

# Test ADF sobre serie original
p_original = adf_test(series, 'Serie original de consumo')

In [ ]:
# Diferenciación si no es estacionaria
series_diff = series.diff().dropna()
p_diff = adf_test(series_diff, 'Serie diferenciada (1er orden)')

# Visualizar antes vs después
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Serie original
axes[0, 0].plot(series.index, series.values, color='#3498db', linewidth=1)
axes[0, 0].set_title('Serie Original', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel(consumption_col, fontsize=10)

# Rolling stats original
rm_orig = series.rolling(7, min_periods=1).mean()
rs_orig = series.rolling(7, min_periods=1).std()
axes[0, 1].plot(series.index, series.values, color='#3498db', alpha=0.3, linewidth=0.5)
axes[0, 1].plot(rm_orig.index, rm_orig.values, color='#e74c3c', linewidth=2, label='Media móvil')
axes[0, 1].plot(rs_orig.index, rs_orig.values, color='#2ecc71', linewidth=2, label='Std móvil')
axes[0, 1].set_title('Estadísticas móviles (original)', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=9)

# Serie diferenciada
axes[1, 0].plot(series_diff.index, series_diff.values, color='#9b59b6', linewidth=1)
axes[1, 0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1, 0].set_title('Serie Diferenciada', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Diferencia', fontsize=10)

# Rolling stats diferenciada
rm_diff = series_diff.rolling(7, min_periods=1).mean()
rs_diff = series_diff.rolling(7, min_periods=1).std()
axes[1, 1].plot(series_diff.index, series_diff.values, color='#9b59b6', alpha=0.3, linewidth=0.5)
axes[1, 1].plot(rm_diff.index, rm_diff.values, color='#e74c3c', linewidth=2, label='Media móvil')
axes[1, 1].plot(rs_diff.index, rs_diff.values, color='#2ecc71', linewidth=2, label='Std móvil')
axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Estadísticas móviles (diferenciada)', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=9)

for ax_row in axes:
    for ax in ax_row:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

fig.suptitle('Estacionariedad: Antes y Después de Diferenciación',
             fontsize=15, fontweight='bold', y=1.02)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"Resumen:")
print(f"  Serie original: p={p_original:.6f} {'(estacionaria)' if p_original < 0.05 else '(NO estacionaria)'}")
print(f"  Serie diferenciada: p={p_diff:.6f} {'(estacionaria)' if p_diff < 0.05 else '(NO estacionaria)'}")

---
## 7. Análisis de degradación de batería

Para vehículos eléctricos e híbridos, la degradación de la batería se manifiesta como
una disminución progresiva del nivel máximo de carga (SOC - State of Charge).
Buscamos tendencias descendentes en `battery_soc_pct`.

In [ ]:
# Cargar perfiles de flota para identificar eléctricos/híbridos
fleet_path = os.path.join(project_root, "data/raw/fleet_profiles.csv")
if os.path.exists(fleet_path):
    df_fleet_profiles = pd.read_csv(fleet_path)
    ev_hybrid = df_fleet_profiles[
        df_fleet_profiles['vehicle_type'].isin(['electrico', 'hibrido'])
    ]['vehicle_id'].tolist()
    print(f"Vehículos eléctricos/híbridos: {len(ev_hybrid)}")
else:
    # Si no hay perfiles, usar todos los vehículos con datos de batería
    ev_hybrid = df_telem['vehicle_id'].unique().tolist()
    print(f"Sin perfiles de flota. Analizando todos los vehículos con datos de batería.")

In [ ]:
# Analizar SOC de batería para vehículos eléctricos/híbridos
if 'battery_soc_pct' in df_daily_vehicle.columns:
    df_ev = df_daily_vehicle[
        df_daily_vehicle['vehicle_id'].isin(ev_hybrid)
    ].copy()

    if len(df_ev) > 0:
        # Seleccionar top vehículos con más datos
        ev_counts = df_ev.groupby('vehicle_id')['date'].count()
        top_ev = ev_counts.nlargest(min(5, len(ev_counts))).index.tolist()

        fig, axes = plt.subplots(len(top_ev), 1, figsize=(15, 4 * len(top_ev)), sharex=True)
        if len(top_ev) == 1:
            axes = [axes]

        degradation_results = {}

        for idx, vid in enumerate(top_ev):
            df_v = df_ev[df_ev['vehicle_id'] == vid].set_index('date').sort_index()
            soc = df_v['battery_soc_pct']
            rm_soc = soc.rolling(7, min_periods=1).mean()

            # Determinar tipo de vehículo
            if os.path.exists(fleet_path):
                vtype = df_fleet_profiles[
                    df_fleet_profiles['vehicle_id'] == vid
                ]['vehicle_type'].values
                vtype = vtype[0] if len(vtype) > 0 else 'desconocido'
            else:
                vtype = 'desconocido'

            color = vtype_colors.get(vtype, '#95a5a6')

            axes[idx].plot(soc.index, soc.values, alpha=0.3, color=color, linewidth=0.8)
            axes[idx].plot(rm_soc.index, rm_soc.values, color=color, linewidth=2.5,
                           label=f'Media móvil 7d')

            # Regresión lineal para detectar degradación
            soc_clean = soc.dropna()
            if len(soc_clean) > 1:
                x = np.arange(len(soc_clean))
                slope, intercept = np.polyfit(x, soc_clean.values, 1)
                trend_line = slope * x + intercept
                axes[idx].plot(soc_clean.index, trend_line, '--', color='black',
                               linewidth=1.5, alpha=0.7, label=f'Tendencia (m={slope:.4f})')
                degradation_results[vid] = slope

            axes[idx].set_ylabel('SOC (%)', fontsize=10)
            axes[idx].set_title(f'Vehículo {vid} ({vtype})', fontsize=11, fontweight='bold')
            axes[idx].legend(fontsize=9, loc='best')

        axes[-1].set_xlabel('Fecha', fontsize=12)
        fig.suptitle('Evolución del Estado de Carga (SOC) - Vehículos Eléctricos/Híbridos',
                     fontsize=14, fontweight='bold', y=1.01)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()

        # Resumen de degradación
        print("\nAnálisis de degradación de batería:")
        print("=" * 50)
        for vid, slope in degradation_results.items():
            if slope < -0.01:
                print(f"  [!] Vehículo {vid}: pendiente={slope:.5f}/día -> POSIBLE DEGRADACIÓN")
            elif slope > 0.01:
                print(f"  [OK] Vehículo {vid}: pendiente={slope:.5f}/día -> SOC estable/mejorando")
            else:
                print(f"  [OK] Vehículo {vid}: pendiente={slope:.5f}/día -> SOC estable")
    else:
        print("No hay datos de vehículos eléctricos/híbridos.")
else:
    print("Columna 'battery_soc_pct' no disponible en los datos.")

---
## 8. Resumen y Conclusiones

### Tendencias de eficiencia encontradas

El análisis de series temporales de la flota reveló:

- **Tendencia general**: La descomposición permitió separar la evolución de largo plazo
  del ruido diario y los patrones semanales.
- **Estacionalidad semanal**: Se observa un patrón cíclico de 7 días en el consumo,
  probablemente asociado a diferencias entre días laborales y fines de semana.
- **Estacionariedad**: El test ADF determina si la serie necesita diferenciación
  antes de aplicar modelos de forecasting (siguiente notebook).

### Vehículos de preocupación

Los vehículos con tendencia creciente en consumo o decreciente en SOC de batería
requieren atención de mantenimiento preventivo:

1. **Degradación de consumo**: Puede indicar problemas mecánicos, necesidad de ajuste,
   o cambio en patrones de uso.
2. **Degradación de batería**: Para vehículos eléctricos/híbridos, una caída sostenida
   del SOC sugiere degradación electroquímica de las celdas.

### Recomendaciones

- Programar inspección para vehículos con tendencia de consumo creciente > 5%.
- Monitorear trimestralmente el SOC máximo de vehículos eléctricos.
- Considerar reemplazo de batería para vehículos con degradación confirmada.

### Siguiente paso

> En el notebook [04 - Forecasting de Series Temporales](04_time_series_forecasting.ipynb)
> usamos modelos predictivos (ARIMA, Prophet) para pronosticar el consumo futuro
> y anticipar necesidades de mantenimiento.